# Machine Learning Assignment Notebook
## Breast Cancer Classification using Four Supervised Learning Algorithms

**Algorithms used**
1. Logistic Regression
2. Decision Tree
3. Random Forest
4. K-Nearest Neighbors (KNN)

**Goal**  
Build and compare four supervised machine learning models to predict whether a breast cancer tumor is **malignant** or **benign**.

**Dataset note**  
This notebook uses the **Breast Cancer Wisconsin dataset** available through `scikit-learn`, which is based on the well-known Wisconsin Diagnostic Breast Cancer dataset.  
For your report, you can cite the dataset source as the Wisconsin Diagnostic Breast Cancer dataset.

> If your lecturer specifically wants a dataset link from a hosting website, you can still use this notebook and mention the published dataset source in your report. The modeling workflow remains the same.


In [ ]:
# Imports
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

pd.set_option("display.max_columns", None)


## 1. Load the Dataset

In [ ]:
# Load dataset from scikit-learn
data = load_breast_cancer()

df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target

# Convert target labels to readable names for analysis
df["diagnosis"] = df["target"].map({0: "malignant", 1: "benign"})

print("Dataset shape:", df.shape)
df.head()


## 2. Dataset Description
The dataset contains numeric features computed from digitized images of breast mass cell nuclei.  
The target variable is:

- **0 = malignant**
- **1 = benign**

Now let's inspect the dataset structure.


In [ ]:
df.info()

In [ ]:
df.describe().T.head(10)

## 3. Check Missing Values

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(10)

## 4. Class Distribution

In [ ]:
class_counts = df["diagnosis"].value_counts()
class_counts


In [ ]:
plt.figure(figsize=(6,4))
class_counts.plot(kind="bar")
plt.title("Class Distribution")
plt.xlabel("Diagnosis")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


## 5. Feature Matrix and Target Vector

In [ ]:
X = df.drop(columns=["target", "diagnosis"])
y = df["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)


## 6. Train-Test Split
We split the data into:
- **80% training**
- **20% testing**


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


## 7. Model Definitions
We use four supervised classification algorithms:
- Logistic Regression
- Decision Tree
- Random Forest
- K-Nearest Neighbors


In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]),

    "Decision Tree": Pipeline([
        ("model", DecisionTreeClassifier(random_state=42, max_depth=5))
    ]),

    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            max_depth=8
        ))
    ]),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ])
}

list(models.keys())


## 8. Train and Evaluate Models
We evaluate each model using:
- Accuracy
- Precision
- Recall
- F1-score


In [ ]:
results = []

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    results.append({
        "Algorithm": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False).reset_index(drop=True)
results_df


In [ ]:
# Show rounded values for report use
results_df_rounded = results_df.copy()
for col in ["Accuracy", "Precision", "Recall", "F1-Score"]:
    results_df_rounded[col] = results_df_rounded[col].round(4)

results_df_rounded


## 9. Accuracy Comparison Chart

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(results_df_rounded["Algorithm"], results_df_rounded["Accuracy"])
plt.title("Accuracy Comparison of Algorithms")
plt.xlabel("Algorithm")
plt.ylabel("Accuracy")
plt.xticks(rotation=15)
plt.ylim(0.85, 1.0)
plt.tight_layout()
plt.show()


## 10. Detailed Classification Reports

In [ ]:
for name, pipeline in models.items():
    y_pred = pipeline.predict(X_test)
    print("=" * 70)
    print(name)
    print("=" * 70)
    print(classification_report(y_test, y_pred, target_names=["malignant", "benign"]))
    print()


## 11. Confusion Matrices

In [ ]:
for name, pipeline in models.items():
    y_pred = pipeline.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["malignant", "benign"])
    disp.plot()
    plt.title(f"Confusion Matrix - {name}")
    plt.tight_layout()
    plt.show()


## 12. Feature Importance (Random Forest)
This helps explain which features influenced predictions the most.


In [ ]:
rf_model = models["Random Forest"].named_steps["model"]
importances = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

importances.head(10)


In [ ]:
top10 = importances.head(10).sort_values(by="Importance")

plt.figure(figsize=(8,6))
plt.barh(top10["Feature"], top10["Importance"])
plt.title("Top 10 Important Features - Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## 13. Final Discussion
Use the generated results table to discuss:

- Which algorithm performed best
- Why some models performed better than others
- Possible limitations
- Future improvements

You can usually say:
- Random Forest often performs very well because it combines many decision trees and reduces overfitting.
- Logistic Regression is simpler and interpretable.
- KNN can perform well after scaling but may be sensitive to the choice of `k`.
- Decision Tree is easy to understand but may overfit if not controlled.

You should compare models using the actual values generated in your notebook.


## 14. Save Results for Report

In [ ]:
results_df_rounded.to_csv("model_results.csv", index=False)
importances.to_csv("random_forest_feature_importance.csv", index=False)

print("Saved:")
print("- model_results.csv")
print("- random_forest_feature_importance.csv")


## 15. Simple Conclusion
This notebook demonstrated a supervised learning approach for classifying breast cancer cases using four machine learning algorithms.  
After preprocessing, training, and testing, the models were compared using standard classification metrics such as accuracy, precision, recall, and F1-score.  
The best-performing model can be identified from the results table and discussed in the final report.
